# Making a phylogenetic tree from back-traced nucleotide sequences

Using a proteins nucleotide sequence over amino acid for construction of a phylogenetic tree has a number of advantages.

As each codon has 3 nucleotides per amino acid, this creates a longer allignment with more phylogenetic signal for tree-building algorithms which improves resolution, especially for recent divergence.

Nucleotide sequences also capture synonymous mutations (those which don't change the amino acid), which provides additional evolutionary information that AA sequences do not. This is useful for studying closely-related proteins which we have here.

This approach is also better for recent evolutionary timescales because NTs mutate faster and accumulate changes more quickly. Thuis will be helpful for resolving relationships between the closely related E. coli and Shigella proteins.

I will be alligning the MAFFT output used for the protein tree *prior* to trimming, as this may interfere with threading the sequences onto the 

The process will go as follows:
Retrieve the coding sequences from NCBI -> thread them onto the protein sequence allignment -> generate new tree with RAxML-NG

## 1. Backtracing nucleotide sequences

There is no working tool currently available, so I will manually backtrace all the proteins in the tree from their protein accession number.

## 2. T-COFFEE to thread nucleotide sequences onto allignment

It turned out that the CDS of each protein was off by one codon, or had an extra nucleotide or two extra that changes the reading frame. The below scripts a) identify the discrepancies for a selected sequence ID and b) corrects the entire set of sequences (it had to drop one sequence in the process)

In [14]:
# TROUBLESHOOTING

# This script reads a corresponding aa and nt sequence and prints any discrepancies

from Bio import SeqIO
from Bio.Seq import Seq

prot_f = "protein_tree/cd-hit/final_for_alignment.fasta"
dna_f = "nt_tree/nt_seq.fasta"
seq_id = "HDT0929104.1"

# 1. Get protein (aligned) sequence and strip gaps
prot_dict = {rec.id: rec.seq for rec in SeqIO.parse(prot_f, "fasta")}
prot_aln = prot_dict[seq_id]
prot_ungapped = str(prot_aln).replace("-", "")

print("Protein length (ungapped):", len(prot_ungapped))

# 2. Get nucleotide sequence
dna_dict = {rec.id: rec.seq for rec in SeqIO.parse(dna_f, "fasta")}
dna = dna_dict[seq_id]

print("DNA length:", len(dna))
print("DNA length % 3:", len(dna) % 3)

# 3. Translate DNA
dna_trans = str(dna.translate(to_stop=False))
print("Translated from DNA (first 60 aa):", dna_trans[:60])
print("Protein (first 60 aa):             ", prot_ungapped[:60])

# 4. Compare residue by residue
min_len = min(len(dna_trans), len(prot_ungapped))
for i in range(min_len):
    if dna_trans[i] != prot_ungapped[i]:
        print(f"First mismatch at position {i+1}: DNA={dna_trans[i]}, PROT={prot_ungapped[i]}")
        break
else:
    print("No mismatch within the overlapping length.")
    if len(dna_trans) != len(prot_ungapped):
        print("But lengths differ:", len(dna_trans), "vs", len(prot_ungapped))

Protein length (ungapped): 529
DNA length: 1590
DNA length % 3: 0
Translated from DNA (first 60 aa): SVDELIIIAHIL*LLTKNIIKIILNGFGHSVE*IFRKFIYFVLNL*FYFILNFTGNQKLV
Protein (first 60 aa):              MMKTITKQPILFTDVPVADLRNSMKQDLNQNLIERLWNKIRDFFLDSDKQKAFKSIHKYI
First mismatch at position 1: DNA=S, PROT=M


In [1]:
# Almost every sequence was off by 1 amino acid, or was in the wrong reading frame.

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

prot_file = "protein_tree/cd-hit/final_for_alignment.fasta"
dna_file = "nt_tree/nt_seq.fasta"
out_file = "nt_tree/nt_seq_fixed.fasta"

# Minimum fraction of identical residues to accept a frame
MIN_IDENTITY = 0.8  # 80%; you can relax to 0.7 if needed

# 1. Read protein alignment (strip gaps)
prot_dict = {}
for rec in SeqIO.parse(prot_file, "fasta"):
    prot_dict[rec.id] = str(rec.seq).replace("-", "")

# 2. Read DNA sequences as full SeqRecords (so we keep headers)
dna_records = list(SeqIO.parse(dna_file, "fasta"))
dna_dict = {rec.id: rec for rec in dna_records}


def find_best_cds(prot_seq, dna_seq, min_identity=MIN_IDENTITY):
    """
    Try all 6 frames, find the window whose translation best matches prot_seq.
    Returns the best CDS nucleotide sequence (string) or None.
    """
    prot_len = len(prot_seq)
    if prot_len == 0:
        return None

    best = None  # (identity, cds_nt, strand, frame, aa_start)

    seq_fwd = Seq(dna_seq)
    seq_rev = seq_fwd.reverse_complement()

    for strand_name, seq in [("+", seq_fwd), ("-", seq_rev)]:
        for frame in range(3):
            trans = str(seq[frame:].translate(to_stop=False))
            if len(trans) < prot_len:
                continue

            # Slide window over this translation
            for aa_start in range(0, len(trans) - prot_len + 1):
                window = trans[aa_start:aa_start + prot_len]
                matches = sum(a == b for a, b in zip(window, prot_seq))
                ident = matches / prot_len

                if best is None or ident > best[0]:
                    # record best so far
                    nt_start = frame + aa_start * 3
                    nt_end = nt_start + prot_len * 3
                    cds_nt = str(seq[nt_start:nt_end])

                    best = (ident, cds_nt, strand_name, frame, aa_start)

    if best is None:
        return None

    ident, cds_nt, strand, frame, aa_start = best
    if ident < min_identity:
        return None

    # Note: cds_nt is in whatever orientation gave the best match.
    return cds_nt


fixed_records = []
dropped_ids = []

for seq_id, prot_seq in prot_dict.items():
    if seq_id not in dna_dict:
        print(f"[WARN] {seq_id} missing from DNA file; skipping.")
        dropped_ids.append(seq_id)
        continue

    orig_rec = dna_dict[seq_id]
    dna_seq = str(orig_rec.seq)
    cds_nt = find_best_cds(prot_seq, dna_seq)

    if cds_nt is None:
        print(f"[WARN] Could not find good CDS match for {seq_id}; dropping.")
        dropped_ids.append(seq_id)
        continue

    # Sanity: length multiple of 3 and matches when translated
    trans = str(Seq(cds_nt).translate(to_stop=False))
    if len(trans) != len(prot_seq):
        print(f"[WARN] Length mismatch even after rescue for {seq_id}; dropping.")
        dropped_ids.append(seq_id)
        continue

    matches = sum(a == b for a, b in zip(trans, prot_seq))
    ident = matches / len(prot_seq)
    if ident < MIN_IDENTITY:
        print(f"[WARN] Identity {ident:.2f} < threshold for {seq_id}; dropping.")
        dropped_ids.append(seq_id)
        continue

    # All good: keep this rescued CDS, but preserve original header
    new_rec = SeqRecord(
        Seq(cds_nt),
        id=orig_rec.id,
        name=orig_rec.name,
        description=orig_rec.description,  # original full header text
    )
    fixed_records.append(new_rec)

print(f"\nKept {len(fixed_records)} sequences; dropped {len(dropped_ids)}.")

SeqIO.write(fixed_records, out_file, "fasta")
print(f"Wrote fixed CDS to: {out_file}")

[WARN] Could not find good CDS match for HBB1443132.1; dropping.

Kept 91 sequences; dropped 1.
Wrote fixed CDS to: nt_tree/nt_seq_fixed.fasta


/opt/anaconda3/envs/tcoffee-env/lib/python3.12/site-packages/Bio/Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [2]:
!t_coffee -other_pg seq_reformat \
    -in protein_tree/mafft/aligned.fasta \
    -in2 nt_tree/nt_seq_fixed.fasta \
    -action +thread_dna_on_prot_aln \
    -output fasta > nt_tree/nt_alligned.fasta
    

3. trimAl

In [1]:
!trimal -in nt_tree/nt_alligned.fasta -out nt_tree/nt_alligned.trimal.fasta -automated1

4. ## RAxML-NG

In [5]:
!raxml-ng --all \
--msa nt_tree/nt_alligned.trimal.fasta \
--model GTR+G \
--prefix nt_tree/raxml/nt_tree \
--bs-trees autoMRE \
--threads 8 \
--seed 2


RAxML-NG v. 1.2.2 released on 30.04.2024 by The Exelixis Lab.
Developed by: Alexey M. Kozlov and Alexandros Stamatakis.
Contributors: Diego Darriba, Tomas Flouri, Benoit Morel, Sarah Lutteropp, Ben Bettisworth, Julia Haag, Anastasis Togkousidis.
Latest version: https://github.com/amkozlov/raxml-ng
Questions/problems/suggestions? Please visit: https://groups.google.com/forum/#!forum/raxml

System: Apple M2, 8 cores, 8 GB RAM

RAxML-NG was called at 25-Nov-2025 23:19:28 as follows:

raxml-ng --all --msa nt_tree/nt_alligned.trimal.fasta --model GTR+G --prefix nt_tree/raxml/nt_tree --bs-trees autoMRE --threads 8 --seed 2

Analysis options:
  run mode: ML tree search + bootstrapping (Felsenstein Bootstrap)
  start tree(s): random (10) + parsimony (10)
  bootstrap replicates: parsimony (max: 1000) + bootstopping (autoMRE, cutoff: 0.030000)
  random seed: 2
  tip-inner: OFF
  pattern compression: ON
  per-rate scalers: OFF
  site repeats: ON
  logLH epsilon: general: 10.000000, brlen-triplet

## Tree annotations

Because the headers are slightly different I need to create new annotations.